### CDC-to-Delta Pipeline: Simple Learning Note

This pipeline processes CDC (Change Data Capture) events and applies them to a Delta target table.

#### Flow
1. **Read CDC events**
2. **Rank events** by employee and sequence number
3. **Keep the latest event** for each employee
4. **Create a temporary SQL view**
5. **MERGE INTO** the Delta target table
6. **Insert, update, or mark employees as deleted**

In [0]:
# filename: cdc-pipeline-with-labeled-output.py | Code Generated by Sidekick is for learning and experimentation purposes only.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

target_table = "silver.employee_current_demo"

# ============================================================
# STEP 1: Create the incoming CDC batch
# ============================================================

raw_data = [
    ("EV101", "E101", "Alice", "1990-01-01", "Chicago", "UPDATE", 105),
    ("EV102", "E101", "Alice", "1990-01-01", "Boston", "UPDATE", 110),
    ("EV201", "E102", "Bob", "1988-04-12", "Seattle", "UPDATE", 205),
    ("EV301", "E103", None, None, None, "DELETE", 300),
    ("EV401", "E104", "David", "1995-06-30", "Austin", "INSERT", 400)
]

columns = [
    "event_id",
    "employee_id",
    "employee_name",
    "dob",
    "city",
    "op_type",
    "sequence_number"
]

batch_df = (
    spark.createDataFrame(raw_data, columns)
    .withColumn("dob", F.to_date("dob"))
    .withColumn("sequence_number", F.col("sequence_number").cast("long"))
    .withColumn("op_type", F.upper(F.col("op_type")))
)

print("===== STEP 1: RAW CDC EVENTS =====")
print("Meaning: All events received in the current batch.")
print(f"Number of raw CDC events: {batch_df.count()}")

display(batch_df.orderBy("employee_id", "sequence_number"))


# ============================================================
# STEP 2: Rank events for each employee
# ============================================================

ranked_events = (
    batch_df
    .withColumn(
        "row_number",
        F.row_number().over(
            Window
            .partitionBy("employee_id")
            .orderBy(
                F.col("sequence_number").desc(),
                F.col("event_id").desc()
            )
        )
    )
)

print("===== STEP 2: RANKED CDC EVENTS =====")
print(
    "Meaning: Events are ranked within each employee. "
    "row_number = 1 means the newest event."
)

display(
    ranked_events
    .select(
        "employee_id",
        "employee_name",
        "city",
        "op_type",
        "sequence_number",
        "row_number"
    )
    .orderBy("employee_id", "row_number")
)


# ============================================================
# STEP 3: Keep only the latest event per employee
# ============================================================

latest_events = (
    ranked_events
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .withColumn(
        "is_deleted",
        F.col("op_type") == F.lit("DELETE")
    )
)

print("===== STEP 3: LATEST EVENT PER EMPLOYEE =====")
print(
    "Meaning: Only the newest CDC event for each employee "
    "will be sent to the MERGE."
)
print(f"Number of latest employee events: {latest_events.count()}")

display(
    latest_events
    .select(
        "employee_id",
        "employee_name",
        "dob",
        "city",
        "op_type",
        "sequence_number",
        "is_deleted"
    )
    .orderBy("employee_id")
)


# ============================================================
# STEP 4: Create the target Delta table
# ============================================================

target_before_df = spark.createDataFrame(
    [
        ("E101", "Alice", "1990-01-01", "New York", 100, False),
        ("E102", "Bob", "1988-04-12", "Chicago", 200, False),
        ("E103", "Carol", "1985-09-09", "Denver", 250, False)
    ],
    """
    employee_id STRING,
    employee_name STRING,
    dob STRING,
    city STRING,
    sequence_number LONG,
    is_deleted BOOLEAN
    """
).withColumn("dob", F.to_date("dob"))

spark.sql(f"DROP TABLE IF EXISTS {target_table}")

(
    target_before_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

print("===== STEP 4: TARGET TABLE BEFORE MERGE =====")
print(
    "Meaning: Current employee data before applying "
    "the incoming CDC events."
)

display(
    spark.table(target_table)
    .orderBy("employee_id")
)


# ============================================================
# STEP 5: Register latest events as a temporary SQL view
# ============================================================

latest_events.createOrReplaceTempView("latest_events")

print("===== STEP 5: TEMPORARY SOURCE VIEW =====")
print(
    "Meaning: latest_events is now available to SQL "
    "as the MERGE source."
)

display(
    spark.sql("""
        SELECT
            employee_id,
            employee_name,
            city,
            op_type,
            sequence_number,
            is_deleted
        FROM latest_events
        ORDER BY employee_id
    """)
)


# ============================================================
# STEP 6: Merge source events into the target Delta table
# ============================================================

print("===== STEP 6: RUNNING MERGE INTO =====")
print(
    "Meaning: Existing employees are updated, new employees "
    "are inserted, and delete events are marked as deleted."
)

spark.sql(f"""
    MERGE INTO {target_table} AS target
    USING latest_events AS source

    ON target.employee_id = source.employee_id

    WHEN MATCHED
      AND source.sequence_number >
          COALESCE(target.sequence_number, CAST(-1 AS BIGINT))

    THEN UPDATE SET
        target.employee_name =
            CASE
                WHEN source.is_deleted
                THEN target.employee_name
                ELSE source.employee_name
            END,

        target.dob =
            CASE
                WHEN source.is_deleted
                THEN target.dob
                ELSE source.dob
            END,

        target.city =
            CASE
                WHEN source.is_deleted
                THEN target.city
                ELSE source.city
            END,

        target.sequence_number = source.sequence_number,
        target.is_deleted = source.is_deleted

    WHEN NOT MATCHED
      AND source.is_deleted = false

    THEN INSERT (
        employee_id,
        employee_name,
        dob,
        city,
        sequence_number,
        is_deleted
    )

    VALUES (
        source.employee_id,
        source.employee_name,
        source.dob,
        source.city,
        source.sequence_number,
        source.is_deleted
    )
""")


# ============================================================
# STEP 7: Display the final target table
# ============================================================

final_df = (
    spark.table(target_table)
    .orderBy("employee_id")
)

print("===== STEP 7: FINAL TARGET TABLE AFTER MERGE =====")
print(
    "Meaning: Final current-state employee data "
    "after applying the latest CDC events."
)

display(final_df)
